# N=96, fall ensemble: the transition becomes measurable, and the seed swamps the grid

**Complete: all six cells COMPLETED exit `0:0`.** `50860571` (array 0-4,
`sud2N96FallScan`, submitted 2026-11-02 19:24:17, last task ended 2026-11-03 10:56:23) and
`50903022` (array 5-5, `sud2N96FallS1L8`, submitted 2026-11-03 04:22:03, ended
2026-11-03 12:18:25), partition `sganguli`, one H100 80GB per cell,
`sh04-02n01`/`sh04-03n01`, ~67 GPU-h in total.

**Commits.** Scan script `c531a86`; the L=8 seed-1 replicate `c71146d`; the reader this
notebook imports, `scripts/n96_fallscan_analysis.py`, committed the same morning.

**Config, identical in every cell except the knob named.** `n_nodes 96, degree 5,
n_colors 4, --fall --shuffle-adj`; `d_model 256, nhead 4, dim_feedforward 1024`, pre-norm;
`lr 1e-3 constant, no warmup, batch 32`; `n_val_instances 300`. The grid is
**equal-gradient-step by construction** -- 240k instances x 50 epochs and 120k x 100
epochs are both 375k steps at batch 32 -- so the data axis is instance *coverage* at
fixed optimization, not more optimization.

**Cells and seeds.**

| task | num_layers | n_train | epochs | seed | role |
|---|---|---|---|---|---|
| t0 | 12 | 240k | 50 | 0 | headline |
| t1 | 8 | 240k | 50 | 0 | depth, shallow |
| t2 | 16 | 240k | 50 | 0 | depth, deep |
| t3 | 12 | 120k | 100 | 0 | data axis, equal steps |
| t4 | 12 | 240k | 50 | **1** | seed replicate at L=12 |
| t5 | 8 | 240k | 50 | **1** | seed replicate at L=8 |

**Why N=96 at all.** At N=48 the fall ensemble drove `rho_min` to 0.332 but the trained
models solved the whole grid: every cell sat above 0.5 at the floor, so `rho_50` was
censored and the scaling question (finding 44) could only be answered in solve-rate
differences, not in a crossing. N=96 pushes the task harder at fixed hint density, and
H1 below is the pre-registered check that this actually bought a crossing.

**Reading direction: `rho_50` is the realized hint density at which solve rate crosses
0.5, and LOWER IS BETTER** -- a lower `rho_50` means the model still solves when given
fewer hints, i.e. it reaches deeper into the slow phase.

In [ ]:
import os, sys, math, json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

REPO = os.path.expanduser("~/spin_gnn")
sys.path.insert(0, REPO)

# The COMMITTED reader. Importing rho_50/rho50_ci rather than reimplementing them
# is the point: the notebook and `python3 scripts/n96_fallscan_analysis.py` cannot
# disagree, and the N=48 crossing convention (first UPWARD crossing of 0.5, walking
# up in rho) carries over so the two scans' numbers mean the same thing.
from scripts.n96_fallscan_analysis import (
    SCAN_DEFAULT, TASKS, load, rho_50, rho50_ci, history)

rows = load(os.path.join(SCAN_DEFAULT, "rho_curve_t*.csv"))
KEY = ("num_layers", "dim_feedforward", "n_train_instances", "seed")
cells = defaultdict(list)
for r in rows:
    cells[tuple(r[k] for k in KEY)].append(r)
for k in cells:
    cells[k].sort(key=lambda r: r["realized_rho"])

LABEL = {v: k for k, v in TASKS.items()}
present = [k for k in TASKS.values() if k in cells]
missing = [LABEL[k] for k in TASKS.values() if k not in cells]

print(f"scan dir : {SCAN_DEFAULT}")
print(f"cells in : {len(present)} of {len(TASKS)}")
for k in present:
    print(f"   {LABEL[k]:<32} {len(cells[k])} rho bins, "
          f"{cells[k][0]['n_instances']} instances/bin")
if missing:
    print("STILL RUNNING / ABSENT:", "; ".join(missing))

## The eval, stated once so every number below is comparable

`scripts/rho_transition_eval` on a 14-point grid `rho = 0.95 ... 0.32`, **400 fresh
instances per bin**, fall ensemble, `--shuffle-adj`. Every cell is evaluated with the
default `--seed-start`, so all cells see the **same 400 instances at each rho** and the
curves are paired point for point -- a difference between two cells is a paired
difference, not two independent samples.

`rho` is the *requested* density; `realized_rho` is what greedy hint removal actually
achieved, and it is the x-axis everywhere below. The fall ensemble's floor is the lowest
realized density reachable at all.

There is **no earlier N=96 transition curve in this project** -- this eval did not exist
before `50860571`. So "previous best on the same eval" below means the best cell that had
landed when the question was asked this morning (t1), not an older run. The N=48 fall
scan is a different eval (different N, different floor) and is not comparable cell to
cell.

In [ ]:
def summarize(k):
    g = cells[k]
    v, flag = rho_50([(r["realized_rho"], r["solve_rate"]) for r in g])
    lo, hi, dropped = rho50_ci(g)
    fv, bv, tr, ne = history(g[0]["path"])
    scored = [r for r in g if r["solve_rate"] > 0.1]     # where well-formedness matters
    return dict(
        label=LABEL[k], L=k[0], n_train=k[2], seed=k[3],
        n_params=g[0]["n_params"], epochs=ne,
        rho50=v, flag=flag, lo=lo, hi=hi,
        floor=min(r["realized_rho"] for r in g),
        val=fv, best_val=bv, train=tr,
        wf_min=min(r["well_formed_rate"] for r in g),
        wf_min_scored=min((r["well_formed_rate"] for r in scored), default=float("nan")),
        tr_max=max(r["truncated_rate"] for r in g),
        tr_max_scored=max((r["truncated_rate"] for r in scored), default=float("nan")),
        T_max=max(r["dd_mean_T"] for r in g))

S = {k: summarize(k) for k in present}

hdr = (f"{'cell':<32}{'L':>3}{'n_train':>9}{'seed':>5}{'params':>10}"
       f"{'rho_50':>9}{'95% CI':>18}{'val_nll':>9}{'wf_min':>8}{'trunc_max':>10}")
print(hdr); print("-" * len(hdr))
for k in present:
    s = S[k]
    ci = f"[{s['lo']:.3f}, {s['hi']:.3f}]" if not math.isnan(s["lo"]) else "n/a"
    star = "" if s["flag"] == "ok" else f" ({s['flag']})"
    print(f"{s['label']:<32}{s['L']:>3}{s['n_train']:>9}{s['seed']:>5}{s['n_params']:>10,}"
          f"{s['rho50']:>9.3f}{ci:>18}{s['val']:>9.4f}{s['wf_min']:>8.4f}"
          f"{s['tr_max']:>10.4f}{star}")

best = min(present, key=lambda k: S[k]["rho50"])
print(f"\nBEST rho_50 (lower is better): {S[best]['rho50']:.3f} -- {S[best]['label']}")
print(f"grid floor (lowest realized rho reachable): {min(S[k]['floor'] for k in present):.4f}")
print(f"max dd_mean_T anywhere: {max(S[k]['T_max'] for k in present):.2f} DD rounds")

## H1 -- PASSES. The transition is measurable at N=96.

> Pre-registered: *"every cell's solve_rate crosses 0.5 strictly inside the rho grid, so
> rho_50 is an interpolation and not a censored bound."*

This is the clause that **failed at N=48**, where every cell sat above 0.5 even at the
floor and `rho_50` could only be reported as `<floor`. At N=96 **all six cells** cross
inside the grid: the flags in the table above are all `ok`, and `dd_mean_T` runs from
about 1.2 rounds at the easy end to >21 rounds at the floor, so the grid really does span
fast phase to deep slow phase.

**This is the result that justifies the move to N=96, and it is the one unambiguous win
in the run.**

## The eval controls, checked before anything is interpreted

In [ ]:
# 1. Pairing: every cell must have hit the same realized densities, or the "paired
#    difference" claim is false and the cells are being compared at different tasks.
ref = [round(r["realized_rho"], 6) for r in cells[present[0]]]
bad = {LABEL[k]: [round(r["realized_rho"], 6) for r in cells[k]]
       for k in present if [round(r["realized_rho"], 6) for r in cells[k]] != ref}
print(f"pairing: {len(present)-len(bad)}/{len(present)} cells on the identical rho grid"
      + ("" if not bad else f"  MISMATCHED: {list(bad)}"))

# 2. Truncation. The max_len margin fix (resolve_max_len_margin -> 194 at N=96) is only
#    load-bearing where the model actually solves; a truncated generation at rho where
#    solve_rate ~ 0 is noise, at rho near the crossing it would corrupt rho_50.
print(f"truncated_rate, bins with solve>0.1: max "
      f"{max(S[k]['tr_max_scored'] for k in present):.4f}  (pre-registered bar 0.01)")
print(f"truncated_rate, all bins          : max "
      f"{max(S[k]['tr_max'] for k in present):.4f}")

# 3. Well-formedness.
print(f"well_formed_rate, all bins        : min "
      f"{min(S[k]['wf_min'] for k in present):.4f}")
for k in present:
    print(f"    {LABEL[k]:<32} min wf {S[k]['wf_min']:.4f}")

## H4 -- FAILS, by more than a factor of two, and it invalidates the rest of the grid

> Pre-registered: *"the seed term, |rho_50(t0) - rho_50(t4)|, is <= 0.05, small against
> the depth and data gaps H2 and H3 are reading. **IF IT IS LARGER THAN 0.05 HERE, this
> whole grid is one draw and only the ordering survives -- say so rather than quoting the
> levels.**"*

There are now **two** such pairs, at both ends of the depth axis the headline claim spans:
t0/t4 at L=12 and t1/t5 at L=8. Each pair is the **same configuration** -- same depth,
same 240k instances, same 50 epochs, every hyper-parameter byte-identical -- differing
only in `--seed`. The cell below measures what that one knob is worth.

Both fail, and the shallow pair fails *harder*: **0.138 at L=8**, against 0.115 at L=12.
That is the opposite of the hopeful reading -- the replicate was submitted precisely in
case the L=8 spread was small enough to rescue H2, and instead it set the yardstick. It
also reproduces finding 41, which measured the big seed term at the shallow end at N=48.

In [ ]:
def seed_pairs():
    byconf = defaultdict(dict)
    for k in present:
        byconf[(k[0], k[1], k[2])][k[3]] = k
    return {c: d for c, d in byconf.items() if len(d) > 1}

pairs = seed_pairs()
sigma_hat = {}
YARD = float("nan")   # defined here so later cells import cleanly if no pair has landed
for conf, d in sorted(pairs.items()):
    ks = [d[s] for s in sorted(d)]
    vals = [S[k]["rho50"] for k in ks]
    spread = max(vals) - min(vals)
    # Paired per-rho solve-rate difference: the curves share instances, so this is the
    # same model evaluated twice, not sampling noise.
    a, b = cells[ks[0]], cells[ks[1]]
    dsolve = [abs(x["solve_rate"] - y["solve_rate"]) for x, y in zip(a, b)]
    # sd from two draws, bias-corrected for n=2 (c4 = sqrt(2/pi) = 0.7979).
    sigma_hat[conf[0]] = (spread / math.sqrt(2)) / 0.79788
    print(f"L={conf[0]:<3} n_train={conf[2]:<7} seeds {sorted(d)}: "
          f"rho_50 {' vs '.join(f'{v:.3f}' for v in vals)}  "
          f"spread {spread:.3f}   sigma_hat {sigma_hat[conf[0]]:.3f}")
    print(f"      per-rho |delta solve_rate|: max {max(dsolve):.3f}  mean {np.mean(dsolve):.3f}")

if sigma_hat:
    # Read a gap against the NOISIER end it spans: a depth claim is only as sharp as the
    # worse-measured of the two rungs it is a difference of.
    YARD = max(s * math.sqrt(2) * 0.79788 for s in sigma_hat.values())
    print(f"\nSEED YARDSTICK (largest measured spread): {YARD:.3f}   pre-registered bar: 0.050")
    print("H4 " + ("PASSES" if YARD <= 0.05 else "FAILS -- levels are not quotable, "
                                                 "only orderings that exceed this"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for conf, d in sorted(pairs.items()):
    for s in sorted(d):
        g = cells[d[s]]
        ax[0].plot([r["realized_rho"] for r in g], [r["solve_rate"] for r in g],
                   "o-", ms=4, label=f"L={conf[0]} seed {s}", rasterized=True)
        v, _ = rho_50([(r["realized_rho"], r["solve_rate"]) for r in g])
        ax[0].axvline(v, ls=":", lw=1, color=ax[0].lines[-1].get_color())
ax[0].axhline(0.5, color="k", lw=0.8, ls="--")
ax[0].set_xlabel("realized rho"); ax[0].set_ylabel("solve rate")
ax[0].set_title("Same config, different seed\n(dotted = rho_50)")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

for conf, d in sorted(pairs.items()):
    ks = [d[s] for s in sorted(d)]
    a, b = cells[ks[0]], cells[ks[1]]
    ax[1].plot([x["realized_rho"] for x in a],
               [x["solve_rate"] - y["solve_rate"] for x, y in zip(a, b)],
               "o-", ms=4, label=f"L={conf[0]}: seed0 - seed1", rasterized=True)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel("realized rho"); ax[1].set_ylabel("paired delta solve_rate")
ax[1].set_title("Paired seed difference at each rho\n(same 400 instances per point)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## H2 and H3 -- both read SMALLER than the seed term, so neither is resolvable

> H2, pre-registered: *"rho_50 falls monotonically with depth, 8 -> 12 -> 16, and the
> 8 -> 12 gap is at least 0.05."*
>
> H3, pre-registered: *"240k beats 120k at equal gradient steps, i.e. coverage and not
> optimization is binding at N=96. Falsifier: if 120k beats 240k, then optimization and
> not coverage is binding at N=96."*

With L=16 in, **H2's ordering is exactly right**: `rho_50` falls monotonically
0.691 -> 0.679 -> 0.613 as depth goes 8 -> 12 -> 16. But the clause H2 actually
pre-registered is the *size* of the 8 -> 12 gap, and that gap is **+0.012** -- a ninth of
the 0.138 seed yardstick. The 12 -> 16 step is larger at 0.066 and still only half the
yardstick. So the direction is encouraging and **the magnitude is not a measurement**:
three draws landing in one particular order happens 1 time in 6 by chance, and the gaps
here are smaller than the noise that would produce that.

H3's falsifier fires *nominally* -- 120k lands below 240k at equal gradient steps -- but
by 0.069, also under the yardstick. Neither clause survives H4.

In [ ]:
print("H2, the depth axis at 240k / seed 0:")
dep = sorted((k for k in present if k[2] == 240000 and k[3] == 0), key=lambda k: k[0])
for k in dep:
    print(f"   L={k[0]:<3} rho_50 {S[k]['rho50']:.3f}  [{S[k]['lo']:.3f}, {S[k]['hi']:.3f}]"
          f"   val_nll {S[k]['val']:.4f}   params {S[k]['n_params']:,}")
for a, b in zip(dep, dep[1:]):
    gap = S[a]["rho50"] - S[b]["rho50"]
    print(f"   gap L={a[0]}->L={b[0]}: {gap:+.3f}   "
          f"{'BELOW' if abs(gap) < YARD else 'above'} the {YARD:.3f} seed yardstick"
          f"   ({'helps' if gap > 0 else 'hurts'})")

print("\nH3, the data axis at L=12 / seed 0 (equal 375k gradient steps):")
dat = sorted((k for k in present if k[0] == 12 and k[3] == 0), key=lambda k: k[2])
for k in dat:
    print(f"   n_train={k[2]:<7} epochs {S[k]['epochs']:<4} rho_50 {S[k]['rho50']:.3f}"
          f"  [{S[k]['lo']:.3f}, {S[k]['hi']:.3f}]   val_nll {S[k]['val']:.4f}")
if len(dat) == 2:
    gap = S[dat[1]]["rho50"] - S[dat[0]]["rho50"]
    print(f"   240k - 120k = {gap:+.3f}  ->  "
          f"{'120k wins (falsifier direction)' if gap > 0 else '240k wins (H3 direction)'}")
    print(f"   |{gap:.3f}| is {'BELOW' if abs(gap) < YARD else 'above'} the "
          f"{YARD:.3f} seed yardstick -> {'NOT RESOLVABLE' if abs(gap) < YARD else 'resolvable'}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for k in present:
    g = cells[k]
    ls = "--" if k[3] == 1 else "-"
    ax.plot([r["realized_rho"] for r in g], [r["solve_rate"] for r in g],
            ls, marker="o", ms=4, label=LABEL[k], rasterized=True)
ax.axhline(0.5, color="k", lw=0.8, ls="--")

# The seed band, drawn at the headline cell so depth gaps are read against it.
if pairs:
    hl = (12, 1024, 240000, 0)
    if hl in S:
        c = S[hl]["rho50"]
        ax.axvspan(c - YARD / 2, c + YARD / 2, color="grey", alpha=0.18, zorder=0,
                   label=f"seed yardstick ({YARD:.3f} wide)")
ax.set_xlabel("realized rho"); ax.set_ylabel("solve rate")
ax.set_title("N=96 fall ensemble: every cell, 400 paired instances per point\n"
             "grey band = what one seed is worth")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## The result that actually explains the grid: rho_50 is a function of val_nll

Finding 4 established that val_nll is a poor predictor of solve rate *across* models
(r = +0.33 over 34 runs), and findings 20 and 25 had it ranking cells backwards. Inside
this grid -- one N, one ensemble, one protocol, varying only depth, data and seed -- it
does not misbehave at all. **Over all six cells the rank correlation is +0.943**, and the
single inversion is t2 vs t3, whose `rho_50` differ by 0.003 with overlapping CIs. Six
cells spanning 0.086 in val_nll span 0.266 in `rho_50`.

This is what H4's failure actually was. **Depth, data and seed are not three separate
effects on reasoning reach -- they are three ways of moving one number, how far the run
converged, and `rho_50` follows that number.** The decisive case is the pair that was
added to rescue H2: t5 is L=8 at seed 1, and it both converged worst (val 1.8140, a full
0.065 behind its own seed-0 twin) and solved worst (0.829). It is not subtly noisy, it is
a partly-failed optimization, and nothing about its *architecture* differs from t1's.

Note the sizes. The seed moves val_nll by up to 0.065 (t1 -> t5). Depth, over the whole
8 -> 16 range, moves it by 0.0175 (t1 -> t2). **The seed outruns depth in the convergence
channel by almost a factor of four**, which is why it outruns it in `rho_50`. Every cell
is still descending when training stops (last-5-epoch val drop negative in all six; the
negative "overfit gap" is finding 16's averaging artifact, not fit).

**Two practical consequences.** (1) `val_nll` is a near-free proxy for `rho_50` inside a
fixed (N, ensemble, protocol) -- the 14-point x 400-instance eval costs hours per cell and
the ranking it produces is already visible in `history.json`. Future scans can screen on
it. (2) Any intervention that lowers val_nll and tightens its spread should move `rho_50`
with it, which is the premise of the follow-up run below.

In [ ]:
print(f"{'cell':<32}{'val_nll':>9}{'best_val':>10}{'rho_50':>9}{'epochs':>8}")
for k in sorted(present, key=lambda k: S[k]["val"]):
    s = S[k]
    print(f"{s['label']:<32}{s['val']:>9.4f}{s['best_val']:>10.4f}{s['rho50']:>9.3f}{s['epochs']:>8}")

v = [S[k]["val"] for k in present]
r = [S[k]["rho50"] for k in present]
if len(present) > 2:
    rv = np.argsort(np.argsort(v)); rr = np.argsort(np.argsort(r))
    n = len(present)
    rho_s = 1 - 6 * np.sum((rv - rr) ** 2) / (n * (n ** 2 - 1))
    print(f"\nSpearman rank correlation val_nll vs rho_50 over {n} cells: {rho_s:+.3f}"
          f"   (n is tiny -- this is an observation, not an estimate)")
    print(f"val_nll range {max(v)-min(v):.4f}   rho_50 range {max(r)-min(r):.3f}")

fig, ax = plt.subplots(figsize=(6, 4.2))
for k in present:
    ax.scatter(S[k]["val"], S[k]["rho50"], s=70, rasterized=True)
    ax.annotate(LABEL[k].split(" (")[0], (S[k]["val"], S[k]["rho50"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel("final val_nll"); ax.set_ylabel("rho_50  (lower = better)")
ax.set_title("Small val_nll differences, large rho_50 differences")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## The well-formedness dip is one cell, and it is the seed again

The grid's minimum `well_formed_rate` is well below the ~1.00 seen when only t1 had
landed. It is **not** spread across cells and it is **not** a depth effect: it is confined
to a single cell, at low rho, and that cell's byte-identical twin at the other seed is
clean everywhere. Truncation is not the cause -- these generations terminate, they just do
not parse. Every cell including t4 is at `well_formed = 1.0000` above rho 0.8.

Three consequences. First, it is a second observable on which one seed changed the model
qualitatively, which corroborates H4 rather than being an independent problem. Second,
malformed output counts as *unsolved*, so it depresses that cell's solve rate -- the
affected cell is the grid's best **despite** the dip, not because of it. Third, and
against the obvious guess: **the badly-converged cell is not the malformed one.** t5 has
the worst val_nll in the grid by a wide margin and its grammar is clean (min 0.9925), so
low-rho malformedness is not simply what undertrained models do here.

In [ ]:
print(f"{'cell':<32}{'min wf':>8}{'at rho':>8}{'wf@crossing':>13}")
for k in present:
    g = cells[k]
    w = min(g, key=lambda r: r["well_formed_rate"])
    near = min(g, key=lambda r: abs(r["realized_rho"] - S[k]["rho50"]))
    print(f"{LABEL[k]:<32}{w['well_formed_rate']:>8.4f}{w['realized_rho']:>8.3f}"
          f"{near['well_formed_rate']:>13.4f}")

worst = min(present, key=lambda k: S[k]["wf_min"])
print(f"\nthe dip, bin by bin -- {LABEL[worst]}:")
for r in cells[worst]:
    if r["well_formed_rate"] < 0.99:
        print(f"   rho {r['realized_rho']:.3f}  well_formed {r['well_formed_rate']:.4f}"
              f"  truncated {r['truncated_rate']:.4f}  solve {r['solve_rate']:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.2))
for k in present:
    g = cells[k]
    ax.plot([r["realized_rho"] for r in g], [r["well_formed_rate"] for r in g],
            "o-", ms=4, label=LABEL[k], rasterized=True)
ax.set_xlabel("realized rho"); ax.set_ylabel("well_formed_rate")
ax.set_title("Grammar holds in every cell but one")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Best rho_50, before and after, on the same eval

The morning's reference point was t1 -- the only cell that had landed, `rho_50 = 0.691`.
The table below is the honest before/after against it on the identical eval.

**The improvement is real as a number and worthless as a method.** t4 differs from t1 in
*two* knobs (depth 8 -> 12 **and** seed 0 -> 1). The one-knob control for depth is t0,
which sits essentially on top of t1 -- so almost none of the gain is depth, and almost all
of it is the seed draw. Nothing was tuned to produce it and nothing known would reproduce
it on a fresh seed.

The completed grid makes that concrete in the least flattering way: **the seed cuts both
ways and the downside is bigger than the upside.** Changing seed 0 -> 1 at L=12 bought
-0.115; the same change at L=8 cost **+0.138**. Had t5 been the cell that landed first
this morning, the "best `rho_50`" on this eval would have been reported as 0.829.

In [ ]:
BEFORE = (8, 1024, 240000, 0)        # t1, the only cell in when the question was asked
if BEFORE in S and present:
    b, a = S[BEFORE], S[best]
    print(f"BEFORE  {b['label']:<32} rho_50 {b['rho50']:.3f}  [{b['lo']:.3f}, {b['hi']:.3f}]")
    print(f"AFTER   {a['label']:<32} rho_50 {a['rho50']:.3f}  [{a['lo']:.3f}, {a['hi']:.3f}]")
    print(f"delta   {a['rho50'] - b['rho50']:+.3f}   "
          f"CIs {'disjoint' if a['hi'] < b['lo'] or b['hi'] < a['lo'] else 'OVERLAP'}")
    ctrl = (12, 1024, 240000, 0)     # t0: same depth change as t4, seed held at 0
    if ctrl in S:
        dep_only = S[ctrl]["rho50"] - b["rho50"]
        tot = a["rho50"] - b["rho50"]
        print(f"\n  attributable to DEPTH (t1 -> t0, seed held at 0): {dep_only:+.3f}")
        print(f"  attributable to SEED  (t0 -> t4, depth held at 12): "
              f"{a['rho50'] - S[ctrl]['rho50']:+.3f}")
        print(f"  -> {100*abs(dep_only)/abs(tot):.0f}% of the move is depth, "
              f"{100*abs(a['rho50']-S[ctrl]['rho50'])/abs(tot):.0f}% is the seed")

## What this costs the planned hyper-parameter scan

The standing plan is a hyper-parameter scan at each N to relate reasoning depth to MLP
depth. H4 sets the price of admission: with a per-cell sd of `sigma_hat`, resolving a gap
`delta` between two rungs at 95% confidence and 80% power needs roughly
`n >= 2 sigma^2 (2.8/delta)^2` seeds **per rung**.

In [ ]:
if sigma_hat:
    sg = max(sigma_hat.values())
    print(f"sigma_hat (per cell, from the seed pair) = {sg:.3f}\n")
    print(f"{'gap to resolve':<34}{'seeds/rung':>12}{'cells':>8}{'GPU-h @ 11h':>13}")
    for name, delta in [("H2's pre-registered bar, 0.05", 0.05),
                        ("the seed spread itself, %.3f" % YARD, YARD),
                        ("observed L=8->12 gap", None)]:
        if delta is None:
            d8 = [k for k in present if k[0] == 8 and k[3] == 0]
            d12 = [k for k in present if k[0] == 12 and k[3] == 0]
            if not (d8 and d12):
                continue
            delta = abs(S[d8[0]]["rho50"] - S[d12[0]]["rho50"])
            name = f"observed L=8->12 gap, {delta:.3f}"
        n = math.ceil(2 * sg ** 2 * (2.8 / delta) ** 2)
        print(f"{name:<34}{n:>12}{2*n:>8}{2*n*11:>13,}")
    print("\nAt ~11 GPU-h per cell, averaging the seed away is not affordable for the")
    print("gaps this grid actually shows. The seed term has to be REDUCED, not averaged.")

### The design conclusion, stated plainly

A depth scan at N=96 run the way this grid was run **cannot answer the depth question at
any affordable seed budget**, and the completed grid closes the one escape route it had:
the spread is not a quirk of L=12, it is larger at L=8. Averaging is the wrong lever.

The val_nll regularity says why and points at the right one. Six cells spanning 0.086 in
val_nll span 0.266 in `rho_50` at rank correlation +0.943; all six are still descending
when training stops; and the seed moves val_nll roughly four times as far as the whole
8 -> 16 depth range does. **The seed is buying convergence, not capacity, and until
convergence is controlled no architectural quantity at this N is measurable.**

So the next single change attacks convergence, not architecture. This grid trains at
**constant lr 1e-3, no warmup, no decay**, which is exactly the setting in which the
end-of-training point is a lottery over where the iterate happened to be sitting. A decay
schedule should lower val_nll and tighten its run-to-run spread, and if the mechanism
above is right, `rho_50`'s spread follows it down.

**The follow-up is running as `50933500`** (`scripts/slurm/sudoku2_n96_cosine_2026-11-03.sbatch`,
submitted before t2 and t5 landed): one knob, `--lr-schedule cosine`, at L=12 / 240k /
50 epochs, seeds 0 and 1, everything else byte-identical to t0/t4, same eval, new dated
directory. Pre-registered there: **P1, the test, is whether the seed spread falls below
0.115**; P2 is val_nll below 1.7283; P3, beating `rho_50` 0.564, is explicitly *not* the
test, because 0.564 is itself a lucky draw. Falsifier: spread >= 0.115 **and** no val_nll
gain, in which case the seed term is intrinsic at N=96 and that is the finding.

One thing the completed grid changes about reading it: P1's bar was set against the L=12
spread of 0.115 because that was all that existed at submission. The yardstick is now
**0.138**, so clearing 0.115 is the stricter of the two tests and P1 stands as written.

### Still open

- The dip's mechanism -- what the malformed generations at low rho actually emit -- is
  uninspected. It is confined to t4, does not affect the controls, is absent above
  rho 0.8, and is *not* explained by undertraining (t5 is far worse converged and clean),
  so it has no account at all yet.
- Whether the val_nll proxy survives an intervention. It holds across depth, data and
  seed at fixed schedule; `50933500` changes the schedule, which is exactly the kind of
  change that could break the relation rather than ride it.
- H2 itself. The ordering 8 > 12 > 16 is right and the 12 -> 16 step (0.066) is the
  largest architectural effect in the grid. If `50933500` tightens the spread, the
  cheapest real test of depth is **L=12 vs L=16 under the schedule**, not a rerun of the
  8 -> 12 gap that failed here.